# 06 — Scaling Laws and Model Selection: Choosing the Right Model for the Job

> **Orbit -1 (Foundations)** · **Difficulty**: Intermediate · **Reading time**: ~25 min

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/foundations/06_scaling_laws_and_model_selection.ipynb)

**Prerequisites**: [00_how_llms_work.ipynb](00_how_llms_work.ipynb), [05_attention_and_context.ipynb](05_attention_and_context.ipynb)

**What you will learn**:
- Scaling laws: how performance relates to parameters, data, and compute
- Chinchilla vs Llama training philosophies
- VRAM budgeting: will your model fit?
- Quantization: 4-bit, 8-bit, 16-bit tradeoffs
- A decision framework for model selection

In [ ]:
# @title Setup — Run this cell first
!pip install -q numpy matplotlib pandas

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})

print('✓ All packages ready')

---
## 1 — The Scaling Hypothesis

In 2020, Kaplan et al. at OpenAI published a landmark finding: **language model loss
follows a power law** with respect to three variables — parameters (*N*), dataset
size (*D*), and compute budget (*C*). The relationship is remarkably clean:

$$L(N) = \left(\frac{N_c}{N}\right)^{\alpha_N}, \quad L(D) = \left(\frac{D_c}{D}\right)^{\alpha_D}, \quad L(C) = \left(\frac{C_c}{C}\right)^{\alpha_C}$$

where $\alpha_N \approx 0.076$, $\alpha_D \approx 0.095$, and $\alpha_C \approx 0.050$.

This means **performance improves predictably** as you scale up. A 10× increase in
parameters gives a consistent reduction in loss. No phase transitions, no plateaus —
just smooth power-law curves across many orders of magnitude.

**Key insight**: Parameters matter more than data *per FLOP*, but both matter. The
original Kaplan scaling laws suggested making models as large as possible and training
them on relatively less data. This advice turned out to be **wrong** — as we'll see
in the next section.

In [ ]:
# Visualize Kaplan-style scaling curves
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Loss vs Parameters
N = np.logspace(6, 11, 200)  # 1M to 100B params
alpha_N, Nc = 0.076, 8.8e13
L_N = (Nc / N) ** alpha_N
axes[0].loglog(N, L_N, 'b-', linewidth=2)
axes[0].set_xlabel('Parameters (N)')
axes[0].set_ylabel('Test Loss')
axes[0].set_title('Loss vs Parameters')

# Known model data points (approximate)
model_params = [125e6, 350e6, 1.3e9, 6.7e9, 13e9, 65e9]
model_names  = ['125M', '350M', '1.3B', '6.7B', '13B', '65B']
model_losses = [(Nc / n) ** alpha_N for n in model_params]
axes[0].scatter(model_params, model_losses, c='red', zorder=5, s=40)
for n, name, loss in zip(model_params, model_names, model_losses):
    axes[0].annotate(name, (n, loss), textcoords='offset points',
                     xytext=(5, 5), fontsize=8)

# Loss vs Dataset size
D = np.logspace(8, 13, 200)  # 100M to 10T tokens
alpha_D, Dc = 0.095, 5.4e13
L_D = (Dc / D) ** alpha_D
axes[1].loglog(D, L_D, 'g-', linewidth=2)
axes[1].set_xlabel('Dataset Size (tokens)')
axes[1].set_ylabel('Test Loss')
axes[1].set_title('Loss vs Data')

# Loss vs Compute
C = np.logspace(15, 25, 200)  # FLOPs
alpha_C, Cc = 0.050, 3.1e8
L_C = (Cc / C) ** alpha_C
axes[2].loglog(C, L_C, 'r-', linewidth=2)
axes[2].set_xlabel('Compute (FLOPs)')
axes[2].set_ylabel('Test Loss')
axes[2].set_title('Loss vs Compute')

plt.tight_layout()
plt.suptitle('Kaplan Scaling Laws (2020)', y=1.03, fontsize=13, fontweight='bold')
plt.show()
print('Note: Straight lines on log-log plots = power law relationships')

---
## 2 — Chinchilla vs Llama: Two Training Philosophies

### The Chinchilla correction

In 2022, Hoffmann et al. (DeepMind) showed that Kaplan's original scaling laws had
a critical flaw: they **under-trained** large models. The compute-optimal recipe — the
"Chinchilla rule" — says:

> **Tokens ≈ 20 × Parameters**

A 10B parameter model should see ~200B tokens. GPT-3 (175B params) was trained on
only 300B tokens — roughly **8.5× under-trained** by this rule.

### The Llama counter-argument

Meta's LLaMA team made a deliberate choice to **over-train** smaller models. Why?
Because Chinchilla optimizes for **training compute**, but in production you pay for
**inference compute** on every single request. A smaller model trained longer is
cheaper to *serve* even if it cost more to *train*.

This shift from "compute-optimal training" to "inference-optimal training" reshaped
the industry. Today, most open models follow the Llama philosophy: train small
models on far more tokens than Chinchilla would recommend.

In [ ]:
# Chinchilla vs Llama training comparison
data = {
    'Model': ['GPT-3', 'Chinchilla', 'LLaMA-1 7B', 'LLaMA-1 13B',
              'LLaMA-1 65B', 'LLaMA-2 7B', 'LLaMA-2 70B',
              'LLaMA-3 8B', 'LLaMA-3 70B'],
    'Params (B)':    [175,   70,  7,   13,  65,  7,   70,  8,   70],
    'Tokens (T)':    [0.3, 1.4, 1.0,  1.0, 1.4, 2.0,  2.0, 15.0, 15.0],
    'Chinchilla (T)': [3.5, 1.4, 0.14, 0.26, 1.3, 0.14, 1.4, 0.16, 1.4],
    'Token/Param Ratio': [1.7, 20, 143, 77, 22, 286, 29, 1875, 214],
    'Philosophy': ['Under-trained', 'Compute-optimal', 'Inference-optimal',
                   'Inference-optimal', 'Near-optimal', 'Inference-optimal',
                   'Inference-optimal', 'Inference-optimal', 'Inference-optimal'],
}
df = pd.DataFrame(data)
print('Training Philosophy Comparison')
print('=' * 80)
print(df.to_string(index=False))
print()
print('Chinchilla rule: Tokens ≈ 20 × Params')
print('LLaMA-3 8B trains on 1,875× its parameter count — nearly 100× the Chinchilla ratio!')

### Why does this matter to you?

When comparing models of the same size, **check how many tokens they were trained on**.
A 7B model trained on 2T tokens will typically outperform one trained on 300B tokens.
The LLaMA-3 models represent the extreme of this philosophy — 8B parameters trained
on 15T tokens, which explains their surprising competitiveness with much larger models.

---
## 3 — VRAM Budgeting: Will Your Model Fit?

Before choosing a model, you need to answer one practical question: **will it fit in
memory?** The fundamental equation is:

$$\text{VRAM} = \underbrace{N \times B_{\text{precision}}}_{\text{model weights}} + \underbrace{\text{KV cache}}_{\text{grows with context}} + \underbrace{\text{overhead}}_{\text{framework, activations}}$$

Where:
- **N** = number of parameters
- **B** = bytes per parameter (2 for FP16, 1 for INT8, 0.5 for INT4)
- **KV cache** = 2 × layers × hidden_dim × context_len × batch_size × B_kv
- **Overhead** ≈ 10–20% of the total for CUDA kernels, framework buffers, etc.

Let's build a calculator that estimates VRAM requirements for any model configuration.

In [ ]:
def estimate_vram_gb(
    params_billion: float,
    precision_bits: int = 16,
    context_length: int = 4096,
    num_layers: int = None,
    hidden_dim: int = None,
    batch_size: int = 1,
    overhead_pct: float = 0.15,
) -> dict:
    """Estimate total VRAM for inference."""
    bytes_per_param = precision_bits / 8
    weight_gb = (params_billion * 1e9 * bytes_per_param) / (1024**3)

    # Rough heuristics for architecture if not provided
    if num_layers is None:
        num_layers = int(params_billion * 4) if params_billion < 15 else int(params_billion * 1.2)
    if hidden_dim is None:
        hidden_dim = int((params_billion * 1e9 / (12 * num_layers)) ** 0.5) * 2

    kv_bytes_per_token = 2 * num_layers * hidden_dim * 2 * (precision_bits / 8)
    kv_cache_gb = (kv_bytes_per_token * context_length * batch_size) / (1024**3)

    subtotal = weight_gb + kv_cache_gb
    overhead_gb = subtotal * overhead_pct
    total = subtotal + overhead_gb

    return {
        'weights_gb': round(weight_gb, 2),
        'kv_cache_gb': round(kv_cache_gb, 2),
        'overhead_gb': round(overhead_gb, 2),
        'total_gb': round(total, 2),
    }

# Quick test
result = estimate_vram_gb(7, precision_bits=16)
print(f'LLaMA-2 7B @ FP16: {result}')

In [ ]:
# VRAM requirement table: model sizes × precisions × GPU fit
models = [
    ('Phi-3 Mini',   3.8),
    ('LLaMA-3 8B',   8.0),
    ('Mistral 7B',   7.0),
    ('LLaMA-2 13B', 13.0),
    ('Mixtral 8x7B', 46.7),
    ('LLaMA-2 70B', 70.0),
]
precisions = [16, 8, 4]
gpus = {
    'T4 (16 GB)': 16,
    'L4 (24 GB)': 24,
    'A100 (40 GB)': 40,
    'A100 (80 GB)': 80,
}

rows = []
for name, params in models:
    for bits in precisions:
        est = estimate_vram_gb(params, precision_bits=bits)
        fits = {gpu: '✓' if est['total_gb'] <= vram else '✗'
                for gpu, vram in gpus.items()}
        rows.append({
            'Model': name,
            'Precision': f'{bits}-bit',
            'VRAM (GB)': est['total_gb'],
            **fits,
        })

df_vram = pd.DataFrame(rows)
print('VRAM Requirements — Inference (batch_size=1, ctx=4096)')
print('=' * 85)
print(df_vram.to_string(index=False))

### Reading the table

Key takeaways:
- **FP16**: You need roughly **2 GB per billion parameters** just for weights
- **INT8**: Cuts memory in half — a 13B model fits on a T4
- **INT4**: Cuts it again — a 70B model *squeezes* into an 80 GB A100
- **KV cache** grows with context length and is often the hidden memory killer

For **Colab Free (T4 with 16 GB)**, your practical ceiling is:
- 7–8B models at INT4/INT8 quantization
- 3–4B models at FP16

This is why quantization isn't optional for most practitioners — it's the only way
to run capable models on accessible hardware.

---
## 4 — Quantization Overview

Quantization reduces the precision of model weights to shrink memory usage and
increase inference speed. Here's the landscape:

| Format | Bits | Bytes/Param | Typical Quality Loss | Use Case |
|--------|------|------------|---------------------|----------|
| FP32   | 32   | 4.0        | None (baseline)     | Training only |
| FP16 / BF16 | 16 | 2.0   | Negligible          | Default inference |
| INT8   | 8    | 1.0        | ~0.5% on benchmarks | Good balance |
| NF4 (QLoRA) | 4 | 0.5    | ~1–2% on benchmarks | Memory-constrained |
| GPTQ/AWQ 4-bit | 4 | 0.5 | ~1–3% on benchmarks | Production serving |
| GGUF Q4_K_M | ~4.8 | 0.6 | ~1% on benchmarks   | CPU + GPU offload |

### The NF4 insight (QLoRA)

Normal Float 4-bit (NF4) is a data type designed specifically for normally-distributed
neural network weights. It provides **information-theoretically optimal** 4-bit
quantization for Gaussian-distributed values. Combined with double quantization
(quantizing the quantization constants), QLoRA achieves near-lossless 4-bit inference.

In [ ]:
# Visualize quantization quality-memory tradeoff
quant_data = {
    'Format': ['FP32', 'FP16/BF16', 'INT8', 'NF4/GPTQ-4', 'Q2_K (2-bit)'],
    'Bits':   [32, 16, 8, 4, 2],
    'Relative_Quality': [100, 99.8, 99.3, 98.2, 93.0],
    'Memory_Factor': [1.0, 0.5, 0.25, 0.125, 0.0625],
}
df_q = pd.DataFrame(quant_data)

fig, ax1 = plt.subplots(figsize=(9, 5))
x = range(len(df_q))
bars = ax1.bar(x, df_q['Relative_Quality'], color=['#2196F3', '#4CAF50',
               '#FF9800', '#F44336', '#9C27B0'], alpha=0.8, width=0.5)
ax1.set_ylim(90, 101)
ax1.set_ylabel('Relative Quality (%)', fontsize=12)
ax1.set_xticks(x)
ax1.set_xticklabels(df_q['Format'], fontsize=10)

ax2 = ax1.twinx()
ax2.plot(x, df_q['Memory_Factor'], 'ko--', markersize=8, linewidth=2)
ax2.set_ylabel('Relative Memory (1.0 = FP32)', fontsize=12)
ax2.set_ylim(0, 1.15)

for i, (q, m) in enumerate(zip(df_q['Relative_Quality'], df_q['Memory_Factor'])):
    ax1.text(i, q + 0.3, f'{q}%', ha='center', fontsize=9, fontweight='bold')

plt.title('Quantization: Quality vs Memory Tradeoff', fontsize=13, fontweight='bold')
fig.tight_layout()
plt.show()
print('Black line = memory usage. Bars = quality retention.')
print('Sweet spot: 4-bit quantization retains ~98% quality at 1/8 the memory.')

### GGUF format labels explained

When downloading models from HuggingFace, you'll see GGUF filenames like
`llama-3-8b-instruct-Q4_K_M.gguf`. Here's how to decode them:

| Label | Meaning | Quality | Size |
|-------|---------|---------|------|
| Q2_K  | 2-bit, K-quant | Low — noticeable degradation | Smallest |
| Q3_K_S/M/L | 3-bit, small/medium/large | Usable for simple tasks | Very small |
| Q4_K_S | 4-bit, K-quant small | Good — slight quality loss | Small |
| **Q4_K_M** | **4-bit, K-quant medium** | **Best balance** — recommended | **~4.8 bpw** |
| Q5_K_M | 5-bit, K-quant medium | Very good | Medium |
| Q6_K  | 6-bit, K-quant | Near-lossless | Larger |
| Q8_0  | 8-bit | Essentially lossless | Large |

**Rule of thumb**: Start with **Q4_K_M**. Move to Q5_K_M or Q6_K if quality matters
more than speed. Only use Q2/Q3 if you're truly memory-starved.

---
## 5 — Reading Benchmarks (Without Being Fooled)

Public benchmarks are how the community compares models. The three you'll see
most often:

| Benchmark | Measures | Format | Typical Range |
|-----------|----------|--------|---------------|
| **MMLU** | Broad knowledge (57 subjects) | 4-choice MCQ | 25% (random) → 90%+ |
| **HumanEval** | Code generation (Python) | Function completion | 0% → 90%+ |
| **GSM8K** | Grade-school math reasoning | Word problems | 0% → 95%+ |

### What they don't tell you

1. **Benchmark contamination**: Models may have seen test questions during training.
   The LLaMA-3 technical report explicitly acknowledges decontamination challenges.
2. **Prompt sensitivity**: A model's MMLU score can swing 5–10 points depending on
   the exact prompt template, number of few-shot examples, and formatting.
3. **Saturation**: Top models cluster above 85% on MMLU, making differences
   statistically insignificant.
4. **Task mismatch**: MMLU measures multiple-choice ability. Your application likely
   requires open-ended generation, which is a very different skill.

### The golden rule

> **YOUR evaluation on YOUR task > any public benchmark.**

Build a small eval set (50–100 examples) that represents your actual use case.
Run every candidate model through it. The model that wins on *your* data is the
right model, regardless of what the leaderboard says.

In [ ]:
# Benchmark comparison for popular models
bench = {
    'Model': ['GPT-4o', 'Claude 3.5 Sonnet', 'LLaMA-3 70B',
              'LLaMA-3 8B', 'Mistral 7B', 'Phi-3 Mini (3.8B)',
              'Gemma-2 9B'],
    'Params':  ['~200B?', '~???', '70B', '8B', '7B', '3.8B', '9B'],
    'MMLU':    [88.7, 88.7, 82.0, 68.4, 62.5, 69.9, 71.3],
    'HumanEval': [90.2, 92.0, 81.7, 62.2, 40.2, 58.5, 54.3],
    'GSM8K':   [95.3, 96.4, 93.0, 79.6, 52.2, 82.7, 76.4],
}
df_bench = pd.DataFrame(bench)
print('Benchmark Scores (approximate, higher = better)')
print('=' * 72)
print(df_bench.to_string(index=False))
print()
print('⚠ These numbers change with prompt format, few-shot count, and evaluation harness.')
print('  Use them for rough comparisons only.')

---
## 6 — Decision Framework: Picking the Right Model

Rather than chasing leaderboard scores, walk through this decision tree:

```
1. VRAM Budget
   └─ What GPU(s) do you have?
      └─ Max model size at INT4 = VRAM × 1.6 (billion params, rough)

2. Latency Requirement
   └─ Real-time (<1s first token)? → Prefer smaller models or quantized
   └─ Batch processing? → Larger models OK, optimize throughput

3. Task Complexity
   └─ Classification / extraction → 3–8B often sufficient
   └─ Multi-step reasoning → 13B+ or strong 8B (LLaMA-3)
   └─ Creative / open-ended → 70B+ or API models

4. Data Sensitivity
   └─ Can data leave your network? → API models (GPT-4, Claude)
   └─ Must stay local? → Open models (LLaMA, Mistral)

5. Finetune Plans
   └─ Will you finetune? → Pick model with good base + QLoRA support
   └─ Prompt-only? → Pick best instruct model in your VRAM budget
```

Let's codify this into a function.

In [ ]:
def recommend_model(
    vram_gb: float,
    latency_sensitive: bool = False,
    task_complexity: str = 'medium',     # 'low', 'medium', 'high'
    data_must_stay_local: bool = True,
    will_finetune: bool = False,
) -> str:
    """Simple model selection heuristic."""
    recommendations = []

    if not data_must_stay_local:
        recommendations.append(
            '🌐 API option: GPT-4o or Claude 3.5 Sonnet — best quality, no GPU needed'
        )

    # Determine max model size at INT4
    max_params_b = vram_gb * 1.6  # rough heuristic

    if max_params_b >= 70:
        tier = '70B'
    elif max_params_b >= 13:
        tier = '13B'
    elif max_params_b >= 7:
        tier = '7-8B'
    elif max_params_b >= 3:
        tier = '3-4B'
    else:
        tier = 'tiny'

    model_map = {
        '70B':  'LLaMA-3 70B (Q4) or Mixtral 8x7B (Q4)',
        '13B':  'LLaMA-2 13B (Q4/Q8) or Mistral 7B (FP16)',
        '7-8B': 'LLaMA-3 8B (Q4/Q8) or Mistral 7B (Q4)',
        '3-4B': 'Phi-3 Mini (Q4/FP16)',
        'tiny': 'Phi-3 Mini (Q2) — limited capability',
    }

    recommendations.append(f'💻 Local option ({vram_gb} GB): {model_map[tier]}')

    if will_finetune:
        ft_max = vram_gb * 0.6  # QLoRA needs extra memory
        if ft_max >= 13:
            recommendations.append('🔧 Finetune: LLaMA-3 8B with QLoRA (FP16 + 4-bit base)')
        elif ft_max >= 7:
            recommendations.append('🔧 Finetune: LLaMA-3 8B or Mistral 7B with QLoRA')
        else:
            recommendations.append('🔧 Finetune: Phi-3 Mini with QLoRA')

    if latency_sensitive and tier in ('70B', '13B'):
        recommendations.append(
            '⚡ Latency note: Consider dropping to 7-8B tier for faster responses'
        )

    return '\n'.join(recommendations)

# Example: Colab Free user with a classification task
print('═' * 60)
print('Scenario: Colab Free (T4 16GB), classification, local data')
print('═' * 60)
print(recommend_model(vram_gb=16, latency_sensitive=True,
                      task_complexity='low', data_must_stay_local=True))
print()
print('═' * 60)
print('Scenario: A100 80GB, complex reasoning, can use APIs')
print('═' * 60)
print(recommend_model(vram_gb=80, latency_sensitive=False,
                      task_complexity='high', data_must_stay_local=False,
                      will_finetune=True))

---
## 7 — When Small Models Win

Bigger isn't always better. There are three common scenarios where a smaller model
outperforms a larger one:

### 1. Well-prompted 8B > Poorly-prompted 70B

Prompt engineering has a larger effect on output quality than most people realize.
A carefully crafted system prompt with clear instructions, examples, and constraints
on an 8B model will beat a lazy one-liner on a 70B model for most structured tasks.

### 2. RAG + Small > Large alone

Retrieval-Augmented Generation (RAG) gives a small model access to relevant
knowledge at inference time. A 7B model with the right context retrieved from a
vector database can match or exceed a 70B model relying on parametric memory
alone — especially for domain-specific questions.

### 3. Finetuned 3B > General 70B

For narrow, well-defined tasks (classification, extraction, format conversion),
a finetuned small model dramatically outperforms a general large model. Microsoft's
Phi-3 demonstrated that a 3.8B model trained on high-quality curated data can
rival models 20× its size on reasoning benchmarks.

In [ ]:
# Illustrate: when does model size stop helping?
model_sizes = [3.8, 7, 8, 13, 34, 70]
# Hypothetical accuracy curves for different approaches
general_prompt = [52, 60, 62, 68, 74, 82]
good_prompt    = [61, 72, 74, 78, 82, 86]
rag_pipeline   = [70, 80, 82, 84, 85, 87]
finetuned      = [85, 89, 90, 91, 91, 92]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(model_sizes, general_prompt, 'o--', label='Basic prompt', color='#ccc', linewidth=2)
ax.plot(model_sizes, good_prompt,    's-',  label='Engineered prompt', color='#2196F3', linewidth=2)
ax.plot(model_sizes, rag_pipeline,   '^-',  label='RAG + good prompt', color='#4CAF50', linewidth=2)
ax.plot(model_sizes, finetuned,      'D-',  label='Finetuned (QLoRA)', color='#F44336', linewidth=2)

ax.axhline(y=82, color='gray', linestyle=':', alpha=0.5)
ax.annotate('70B basic prompt = 8B RAG', xy=(34, 82), fontsize=9,
            color='gray', style='italic')

ax.set_xlabel('Model Size (Billion Parameters)', fontsize=12)
ax.set_ylabel('Task Accuracy (%)', fontsize=12)
ax.set_title('Bigger Isn\'t Always Better', fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.set_ylim(45, 95)
plt.tight_layout()
plt.show()
print('A finetuned 3.8B model outperforms a 70B model with a basic prompt.')
print('Invest in prompting/RAG/finetuning before scaling up model size.')

---
## 8 — Colab Pro Cost Estimation

For this course, you'll primarily use Google Colab. Here's a practical breakdown of
what's available and what it costs:

| Tier | GPU Options | VRAM | Approx. Cost | Max Model (INT4) |
|------|------------|------|-------------|------------------|
| Free | T4 | 16 GB | $0/mo | ~8B |
| Pro ($10/mo) | T4, L4 | 16–24 GB | ~$0.10/hr | ~13B |
| Pro+ ($50/mo) | T4, L4, A100 | 16–80 GB | ~$0.50/hr | ~70B |
| Enterprise | A100, H100 | 40–80 GB | Varies | 70B+ |

**For this course**: Colab Free (T4) is sufficient for all notebooks. We design
every exercise to fit within 16 GB VRAM.

In [ ]:
def estimate_colab_cost(
    hours_per_week: float,
    weeks: int,
    tier: str = 'free',  # 'free', 'pro', 'pro_plus'
) -> dict:
    """Estimate Colab costs for the course."""
    tier_info = {
        'free':     {'sub': 0,  'gpu_hr': 0.0,   'gpu': 'T4',  'vram': 16},
        'pro':      {'sub': 10, 'gpu_hr': 0.10,  'gpu': 'L4',  'vram': 24},
        'pro_plus': {'sub': 50, 'gpu_hr': 0.50,  'gpu': 'A100','vram': 80},
    }
    info = tier_info[tier]
    total_hours = hours_per_week * weeks
    compute_cost = total_hours * info['gpu_hr']
    subscription = info['sub'] * (weeks / 4.33)  # months
    total = compute_cost + subscription

    return {
        'tier': tier,
        'gpu': info['gpu'],
        'vram_gb': info['vram'],
        'total_hours': total_hours,
        'compute_cost': f'${compute_cost:.2f}',
        'subscription': f'${subscription:.2f}',
        'total_cost': f'${total:.2f}',
    }

# Course estimate: 4 hours/week for 12 weeks
print('Course Cost Projections (4 hrs/week × 12 weeks)')
print('=' * 55)
for tier in ['free', 'pro', 'pro_plus']:
    est = estimate_colab_cost(4, 12, tier)
    print(f"{tier:10s} | GPU: {est['gpu']:5s} | VRAM: {est['vram_gb']:2d} GB | "
          f"Total: {est['total_cost']}")

print()
print('Recommendation: Start with Free tier. Upgrade to Pro only if you need')
print('longer sessions or want to experiment with 13B+ models.')

---
## Exercises

### Exercise 1: VRAM Budget for 3 Models

Use the `estimate_vram_gb()` function to calculate VRAM requirements for:
1. **Mistral 7B** at FP16, INT8, and INT4
2. **LLaMA-3 70B** at INT4 with 8K context
3. **Phi-3 Mini (3.8B)** at FP16 with 128K context

Which ones fit on a T4 (16 GB)? Which require an A100?

In [ ]:
# Exercise 1 — Your code here
# Hint: estimate_vram_gb(params_billion, precision_bits, context_length)

# 1. Mistral 7B at three precisions
# for bits in [16, 8, 4]:
#     result = estimate_vram_gb(7, precision_bits=bits)
#     print(f'Mistral 7B @ {bits}-bit: {result["total_gb"]} GB')

# 2. LLaMA-3 70B at INT4 with 8K context
# result = estimate_vram_gb(70, precision_bits=4, context_length=8192)

# 3. Phi-3 Mini at FP16 with 128K context
# result = estimate_vram_gb(3.8, precision_bits=16, context_length=131072)

### Exercise 2: Model Selection for a Use Case

You're building a customer support chatbot with these constraints:
- Must run on-premise (data sensitivity)
- Available hardware: 2× NVIDIA L4 (24 GB each)
- Latency: < 500ms first token
- Task: Answer questions using a knowledge base (RAG)

Use the decision framework to recommend a model. Justify your choice.

In [ ]:
# Exercise 2 — Your analysis here
# Use recommend_model() as a starting point, then refine your reasoning.

# print(recommend_model(
#     vram_gb=24,
#     latency_sensitive=True,
#     task_complexity='medium',
#     data_must_stay_local=True,
#     will_finetune=False,
# ))

# Your reasoning:
# - VRAM: ...
# - Latency: ...
# - RAG means: ...
# - Recommended model: ...

### Exercise 3: Cost Projection

You plan to spend 6 hours per week over 16 weeks developing a prototype. Compare
the cost of Colab Free, Pro, and Pro+ tiers. At what point does renting a dedicated
GPU server (e.g., Lambda Labs at ~$1.10/hr for an A100) become more cost-effective
than Colab Pro+?

In [ ]:
# Exercise 3 — Your calculations here
# for tier in ['free', 'pro', 'pro_plus']:
#     est = estimate_colab_cost(6, 16, tier)
#     print(f"{tier}: {est['total_cost']}")

# Lambda Labs comparison:
# lambda_hourly = 1.10
# total_hours = 6 * 16
# lambda_cost = total_hours * lambda_hourly
# print(f'Lambda A100: ${lambda_cost:.2f}')

---
## Key Takeaways

1. **Scaling laws are predictable**: Loss follows power laws with parameters, data,
   and compute. More of each helps, but the returns are logarithmic.

2. **Chinchilla vs Llama**: Compute-optimal training (20× tokens per param) gives
   the best model *per training dollar*. Inference-optimal training (100×+) gives
   the best model *per serving dollar*. The industry chose inference-optimal.

3. **VRAM is your hard constraint**: At FP16, budget ~2 GB per billion parameters.
   Quantization (INT4/NF4) cuts this to ~0.5 GB per billion.

4. **Quantization is nearly free quality-wise**: 4-bit quantization retains ~98%
   of model quality while using 1/4 the memory of FP16. Always consider it.

5. **Benchmarks are noisy guides**: Use MMLU/HumanEval/GSM8K for rough comparisons.
   Build your own eval set for real decisions.

6. **Smaller + smarter often wins**: Good prompts, RAG, or finetuning on a small
   model can beat a large general model. Invest in technique before scale.

## References & Further Reading

1. [Kaplan et al., "Scaling Laws for Neural Language Models," 2020](https://arxiv.org/abs/2001.08361) — The original OpenAI scaling laws paper establishing power-law relationships between loss and compute/data/parameters.
2. [Hoffmann et al., "Training Compute-Optimal Large Language Models" (Chinchilla), 2022](https://arxiv.org/abs/2203.15556) — DeepMind's correction showing models should be trained on ~20× more tokens than Kaplan suggested.
3. [Touvron et al., "LLaMA: Open and Efficient Foundation Language Models," 2023](https://arxiv.org/abs/2302.13971) — Meta's LLaMA-1 paper introducing inference-optimal training philosophy.
4. [Dettmers et al., "QLoRA: Efficient Finetuning of Quantized Language Models," 2023](https://arxiv.org/abs/2305.14314) — The QLoRA paper introducing NF4 quantization and enabling 4-bit finetuning.
5. [Touvron et al., "Llama 2: Open Foundation and Fine-Tuned Chat Models," 2023](https://arxiv.org/abs/2307.09288) — LLaMA-2 with RLHF and extended training.
6. [Meta AI, "The Llama 3 Herd of Models," 2024](https://arxiv.org/abs/2407.21783) — LLaMA-3 with extreme over-training (15T tokens for 8B params).
7. [Hendrycks et al., "Measuring Massive Multitask Language Understanding" (MMLU), 2021](https://arxiv.org/abs/2009.03300) — The MMLU benchmark paper.